# Notebook 1: Data Ingestion — Bronze Layer
## Proyek Analisis Performa Kampanye Marketing
### Big Data dan Analitik (CSD60707)

**Penanggung Jawab:** Esa Khafidotul Khusna Rois (Data Engineer 1 - Data Ingestion)

---

### Medallion Architecture — Bronze Layer

Notebook ini mengimplementasikan **Bronze Layer** dalam Medallion Architecture:

```
┌─────────────┐     ┌──────────────────────────────┐
│   Kaggle     │────▶│   MinIO Data Lake             │
│  (Sumber)    │     │                                │
│              │     │   📁 marketing-data/           │
│              │     │   ├── bronze/  ◄── Layer ini   │
│              │     │   ├── silver/                  │
│              │     │   ├── gold/                    │
│              │     │   └── models/                  │
└─────────────┘     └──────────────────────────────┘
```

**Bronze Layer** = Data mentah (raw) yang diingest **apa adanya** ke Data Lake.
- Tidak ada transformasi atau pembersihan data
- Format asli (CSV) dipertahankan
- Berfungsi sebagai *single source of truth*

### Tujuan Notebook:
1. Memverifikasi dataset Marketing Campaign yang telah diunduh dari Kaggle
2. Melakukan eksplorasi awal data (quick peek)
3. Setup koneksi ke MinIO sebagai Data Lake
4. Membuat bucket dan struktur Medallion (bronze/silver/gold/models)
5. Upload dataset CSV ke MinIO Bronze Layer
6. Verifikasi integritas data setelah ingestion

---
## Step 1: Import Library

In [14]:
import os, sys, warnings, glob as _glob
warnings.filterwarnings('ignore')

# ============================================================
# FIX Windows: Set environment variables sebelum import PySpark
# ============================================================
os.environ["JAVA_HOME"]   = r"C:\Program Files\Java\jdk-21.0.10"
os.environ["SPARK_HOME"]  = r"C:\spark\spark-4.1.1-bin-hadoop3"
os.environ["HADOOP_HOME"] = r"C:\Users\muham\OneDrive\Dokumen\Dokumen\BigData\bigdata6-marketing-analytics\hadoop"
_py4j = _glob.glob(os.path.join(os.environ["SPARK_HOME"], "python", "lib", "py4j-*.zip"))
sys.path.insert(0, os.path.join(os.environ["SPARK_HOME"], "python"))
if _py4j: sys.path.insert(0, _py4j[0])
# ============================================================
import os
import sys
import hashlib
import pandas as pd
from datetime import datetime
from minio import Minio
from minio.error import S3Error

# Tambahkan parent directory ke path untuk import config
sys.path.insert(0, os.path.abspath('..'))

print('[OK] Library berhasil di-import!')
print(f'   Pandas version  : {pd.__version__}')
print(f'   Waktu eksekusi  : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

[OK] Library berhasil di-import!
   Pandas version  : 3.0.2
   Waktu eksekusi  : 2026-04-29 21:38:15


---
## Step 2: Konfigurasi

In [23]:
# ============================================
# KONFIGURASI - Sesuaikan dengan environment
# ============================================

# MinIO Configuration
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
MINIO_BUCKET = "marketing-data"
MINIO_USE_SSL = False

# Path dataset lokal
LOCAL_CSV_PATH = "../data/raw/marketing_campaign_performance_10000.csv"

# Medallion Architecture — Path di MinIO
MINIO_BRONZE_DIR = "bronze/"
MINIO_SILVER_DIR = "silver/"
MINIO_GOLD_DIR = "gold/"
MINIO_MODELS_DIR = "models/"
MINIO_BRONZE_CSV = "bronze/marketing_campaign_performance.csv"

# Kaggle dataset info
KAGGLE_DATASET = "mirzayasirabdullah07/marketing-campaign-performance-dataset"

print('[OK] Konfigurasi dimuat!')
print(f'   MinIO Endpoint : {MINIO_ENDPOINT}')
print(f'   Bucket         : {MINIO_BUCKET}')
print(f'   Bronze Path    : {MINIO_BRONZE_CSV}')
print(f'   Local CSV      : {LOCAL_CSV_PATH}')

[OK] Konfigurasi dimuat!
   MinIO Endpoint : localhost:9000
   Bucket         : marketing-data
   Bronze Path    : bronze/marketing_campaign_performance.csv
   Local CSV      : ../data/raw/marketing_campaign_performance_10000.csv


---
## Step 3: Verifikasi Dataset Lokal

Dataset diunduh dari Kaggle: [Marketing Campaign Performance Dataset](https://www.kaggle.com/datasets/mirzayasirabdullah07/marketing-campaign-performance-dataset)

Jika dataset belum ada, jalankan:
```bash
python scripts/download_dataset.py
```

In [17]:
# Cek apakah file CSV ada di lokal
if os.path.exists(LOCAL_CSV_PATH):
    file_size = os.path.getsize(LOCAL_CSV_PATH)
    print('[OK] Dataset ditemukan!')
    print(f'   Path   : {os.path.abspath(LOCAL_CSV_PATH)}')
    print(f'   Ukuran : {file_size / 1024:.2f} KB ({file_size:,} bytes)')
else:
    print('[ERROR] Dataset TIDAK ditemukan!')
    print(f'   Expected path: {os.path.abspath(LOCAL_CSV_PATH)}')
    print('\n[INFO] Cara mendapatkan dataset:')
    print('   Option 1: python scripts/download_dataset.py')
    print(f'   Option 2: Download manual dari https://www.kaggle.com/datasets/{KAGGLE_DATASET}')

[OK] Dataset ditemukan!
   Path   : c:\Users\muham\OneDrive\Dokumen\Dokumen\BigData\bigdata6-marketing-analytics\data\raw\marketing_campaign_performance_10000.csv
   Ukuran : 803.17 KB (822,441 bytes)


In [18]:
# Load dataset dengan Pandas untuk eksplorasi awal
df = pd.read_csv(LOCAL_CSV_PATH)

print('=' * 60)
print('INFORMASI DATASET')
print('=' * 60)
print(f'Jumlah Record  : {df.shape[0]:,} baris')
print(f'Jumlah Kolom   : {df.shape[1]} kolom')
print(f'Memory Usage   : {df.memory_usage(deep=True).sum() / 1024:.2f} KB')
print(f'\nKolom-kolom:')
for i, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    null_count = df[col].isnull().sum()
    print(f'   {i:2d}. {col:<15s} | {str(dtype):<10s} | Null: {null_count}')

# Hitung MD5 checksum untuk verifikasi integritas
with open(LOCAL_CSV_PATH, 'rb') as f:
    local_md5 = hashlib.md5(f.read()).hexdigest()
print(f'\nMD5 Checksum   : {local_md5}')

INFORMASI DATASET
Jumlah Record  : 10,000 baris
Jumlah Kolom   : 11 kolom
Memory Usage   : 2810.97 KB

Kolom-kolom:
    1. CampaignID      | str        | Null: 0
    2. StartDate       | str        | Null: 0
    3. EndDate         | str        | Null: 0
    4. Channel         | str        | Null: 0
    5. Impressions     | int64      | Null: 0
    6. Clicks          | int64      | Null: 0
    7. Leads           | int64      | Null: 0
    8. Conversions     | int64      | Null: 0
    9. Cost_USD        | float64    | Null: 0
   10. Revenue_USD     | float64    | Null: 0
   11. ROI             | float64    | Null: 0

MD5 Checksum   : e275e6a4db0a0adf9263c445ff191bad


In [19]:
# Tampilkan 5 baris pertama
print('Preview Data (5 baris pertama):')
df.head()

Preview Data (5 baris pertama):


,CampaignID,StartDate,EndDate,Channel,Impressions,Clicks,Leads,Conversions,Cost_USD,Revenue_USD,ROI
0,CAMP00001,2025-04-13,2025-04-19,Search,293520,23335,11643,5389,1052.39,2236.02,1.12
1,CAMP00002,2025-12-15,2025-12-24,Search,200340,15841,6601,2498,3964.90,11740.15,1.96
2,CAMP00003,2025-09-28,2025-10-06,Email,239365,16478,8043,3397,1000.39,1902.24,0.90
3,CAMP00004,2025-04-17,2025-04-30,Search,156382,2672,1014,342,1252.63,2209.74,0.76
4,CAMP00005,2025-03-13,2025-03-22,Influencer,285472,4155,1521,565,4935.48,14111.31,1.86


In [20]:
# Statistik deskriptif
print('Statistik Deskriptif:')
df.describe()

Statistik Deskriptif:


,Impressions,Clicks,Leads,Conversions,Cost_USD,Revenue_USD,ROI
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,152260.637900,8338.993300,2512.463600,1010.700000,2552.356991,5102.648930,1.001576
std,85251.707621,6546.326431,2321.031302,1019.402613,1427.694545,3309.326705,0.578806
min,5043.000000,72.000000,10.000000,4.000000,100.230000,113.640000,0.000000
25%,78956.000000,2981.000000,779.000000,295.000000,1319.622500,2385.742500,0.500000
50%,151505.500000,6588.500000,1777.000000,670.000000,2529.675000,4659.365000,0.990000
75%,225853.250000,12404.250000,3550.250000,1391.000000,3821.307500,7241.245000,1.510000
max,299997.000000,29578.000000,13833.000000,7987.000000,4998.420000,14871.340000,2.000000


In [21]:
# Distribusi Channel
print('Distribusi Channel:')
channel_dist = df['Channel'].value_counts()
for channel, count in channel_dist.items():
    pct = count / len(df) * 100
    print(f'   {channel:<12s}: {count:,} kampanye ({pct:.1f}%)')

print(f'\nDistribusi ROI (Label Klasifikasi):')
profitable = (df['ROI'] >= 1.0).sum()
not_profitable = (df['ROI'] < 1.0).sum()
print(f'   Profitable (ROI >= 1.0)    : {profitable:,} ({profitable/len(df)*100:.1f}%)')
print(f'   Not Profitable (ROI < 1.0) : {not_profitable:,} ({not_profitable/len(df)*100:.1f}%)')

Distribusi Channel:
   Display     : 2,069 kampanye (20.7%)
   Influencer  : 2,056 kampanye (20.6%)
   Email       : 1,996 kampanye (20.0%)
   Social      : 1,955 kampanye (19.6%)
   Search      : 1,924 kampanye (19.2%)

Distribusi ROI (Label Klasifikasi):
   Profitable (ROI >= 1.0)    : 4,987 (49.9%)
   Not Profitable (ROI < 1.0) : 5,013 (50.1%)


---
## Step 4: Koneksi ke MinIO

MinIO berfungsi sebagai **Data Lake** (Object Storage S3-compatible).

Pastikan MinIO server sudah berjalan:
```bash
# Via Docker Compose:
docker-compose up -d minio
```
- API: http://localhost:9000
- Console: http://localhost:9001
- Credentials: minioadmin / minioadmin123

In [24]:
# Buat koneksi ke MinIO
try:
    minio_client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=MINIO_USE_SSL
    )
    
    # Test koneksi dengan list buckets
    buckets = minio_client.list_buckets()
    print('[OK] Berhasil terhubung ke MinIO!')
    print(f'   Endpoint : {MINIO_ENDPOINT}')
    print(f'   Buckets  : {len(buckets)} bucket ditemukan')
    for b in buckets:
        print(f'              - {b.name} (created: {b.creation_date})')
        
except Exception as e:
    print('[ERROR] Gagal terhubung ke MinIO!')
    print(f'   Error: {e}')
    print('\n[INFO] Pastikan MinIO server sudah berjalan:')
    print('   docker-compose up -d minio')

[OK] Berhasil terhubung ke MinIO!
   Endpoint : localhost:9000
   Buckets  : 1 bucket ditemukan
              - marketing-data (created: 2026-04-29 14:10:08.073000+00:00)


---
## Step 5: Buat Bucket & Struktur Medallion Architecture

Struktur penyimpanan di MinIO mengikuti **Medallion Architecture**:
```
marketing-data/              <- Bucket
├── bronze/                   <- Data mentah (CSV) — raw ingestion
├── silver/                   <- Data bersih & enriched (Parquet)
├── gold/                     <- Agregasi bisnis & predictions (Parquet)
└── models/                   <- Model ML tersimpan
```

| Layer | Deskripsi | Format | Contoh |
|-------|-----------|--------|--------|
| **Bronze** | Data mentah, tanpa transformasi | CSV | Raw dataset dari Kaggle |
| **Silver** | Data bersih, validated, enriched | Parquet | Cleaned + feature engineering |
| **Gold** | Business-ready aggregation | Parquet | Channel summary, predictions |
| **Models** | Artefak model ML | Spark ML | Random Forest, Linear Regression |

In [25]:
import io

# Buat bucket jika belum ada
try:
    if not minio_client.bucket_exists(MINIO_BUCKET):
        minio_client.make_bucket(MINIO_BUCKET)
        print(f'[OK] Bucket "{MINIO_BUCKET}" berhasil dibuat!')
    else:
        print(f'[INFO] Bucket "{MINIO_BUCKET}" sudah ada.')
    
    # Buat placeholder untuk struktur Medallion Architecture
    medallion_dirs = [
        (MINIO_BRONZE_DIR, 'Bronze — Data mentah'),
        (MINIO_SILVER_DIR, 'Silver — Data bersih & enriched'),
        (MINIO_GOLD_DIR, 'Gold — Agregasi bisnis'),
        (MINIO_MODELS_DIR, 'Models — Artefak ML'),
    ]
    
    for d, desc in medallion_dirs:
        objects = list(minio_client.list_objects(MINIO_BUCKET, prefix=d))
        if not objects:
            minio_client.put_object(
                MINIO_BUCKET, 
                d + ".gitkeep",
                io.BytesIO(b""),
                0
            )
            print(f'   [OK] {d:<10s} dibuat — {desc}')
        else:
            print(f'   [INFO] {d:<10s} sudah ada — {desc}')
    
    print(f'\n[OK] Struktur Medallion Architecture di bucket "{MINIO_BUCKET}" siap!')

except S3Error as e:
    print(f'[ERROR] Error saat membuat bucket: {e}')

[INFO] Bucket "marketing-data" sudah ada.
   [OK] bronze/    dibuat — Bronze — Data mentah
   [OK] silver/    dibuat — Silver — Data bersih & enriched
   [OK] gold/      dibuat — Gold — Agregasi bisnis
   [INFO] models/    sudah ada — Models — Artefak ML

[OK] Struktur Medallion Architecture di bucket "marketing-data" siap!


---
## Step 6: Upload Dataset ke Bronze Layer

Data di-upload **apa adanya** tanpa transformasi ke Bronze Layer.
Ini menjadi *single source of truth* untuk seluruh pipeline.

In [26]:
# Upload CSV ke Bronze Layer di MinIO
try:
    print('[...] Mengupload dataset ke Bronze Layer...')
    print(f'   Source : {os.path.abspath(LOCAL_CSV_PATH)}')
    print(f'   Target : s3a://{MINIO_BUCKET}/{MINIO_BRONZE_CSV}')
    
    result = minio_client.fput_object(
        bucket_name=MINIO_BUCKET,
        object_name=MINIO_BRONZE_CSV,
        file_path=LOCAL_CSV_PATH,
        content_type="text/csv"
    )
    
    print(f'\n[OK] Upload ke Bronze Layer berhasil!')
    print(f'   Object  : {result.object_name}')
    print(f'   ETag    : {result.etag}')
    
except S3Error as e:
    print(f'[ERROR] Error saat upload: {e}')

[...] Mengupload dataset ke Bronze Layer...
   Source : c:\Users\muham\OneDrive\Dokumen\Dokumen\BigData\bigdata6-marketing-analytics\data\raw\marketing_campaign_performance_10000.csv
   Target : s3a://marketing-data/bronze/marketing_campaign_performance.csv

[OK] Upload ke Bronze Layer berhasil!
   Object  : bronze/marketing_campaign_performance.csv
   ETag    : e275e6a4db0a0adf9263c445ff191bad


---
## Step 7: Verifikasi Ingestion

In [27]:
# List semua objects di bucket
print('Daftar Objects di MinIO Bucket (Medallion Structure):')
print('=' * 65)
objects = minio_client.list_objects(MINIO_BUCKET, recursive=True)
total_size = 0
for obj in objects:
    size_kb = obj.size / 1024 if obj.size else 0
    total_size += obj.size if obj.size else 0
    # Tentukan layer berdasarkan prefix
    if obj.object_name.startswith('bronze/'):
        layer = '[BRONZE]'
    elif obj.object_name.startswith('silver/'):
        layer = '[SILVER]'
    elif obj.object_name.startswith('gold/'):
        layer = '[GOLD]  '
    elif obj.object_name.startswith('models/'):
        layer = '[MODEL] '
    else:
        layer = '[OTHER] '
    print(f'   {layer} {obj.object_name:<50s} {size_kb:>8.2f} KB')
print('=' * 65)
print(f'   Total: {total_size / 1024:.2f} KB')

Daftar Objects di MinIO Bucket (Medallion Structure):
   [BRONZE] bronze/.gitkeep                                        0.00 KB
   [BRONZE] bronze/marketing_campaign_performance.csv            803.17 KB
   [GOLD]   gold/.gitkeep                                          0.00 KB
   [MODEL]  models/.gitkeep                                        0.00 KB
   [SILVER] silver/.gitkeep                                        0.00 KB
   Total: 803.17 KB


In [28]:
import tempfile

print('[...] Verifikasi Integritas Data...')

# Download file dari MinIO ke temp
temp_path = os.path.join(tempfile.gettempdir(), 'verify_download.csv')
minio_client.fget_object(MINIO_BUCKET, MINIO_BRONZE_CSV, temp_path)

# Hitung MD5 file yang didownload
with open(temp_path, 'rb') as f:
    minio_md5 = hashlib.md5(f.read()).hexdigest()

print(f'   MD5 Lokal  : {local_md5}')
print(f'   MD5 MinIO  : {minio_md5}')

if local_md5 == minio_md5:
    print('\n[OK] INTEGRITAS DATA TERJAGA 100%! Checksum cocok.')
else:
    print('\n[WARN] Checksum tidak cocok! Data mungkin corrupt.')

# Verifikasi jumlah record
df_verify = pd.read_csv(temp_path)
print(f'\n   Record lokal : {len(df):,}')
print(f'   Record MinIO : {len(df_verify):,}')
print(f'   Match        : {"[OK] Ya" if len(df) == len(df_verify) else "[ERROR] Tidak"}')

os.remove(temp_path)

[...] Verifikasi Integritas Data...
   MD5 Lokal  : e275e6a4db0a0adf9263c445ff191bad
   MD5 MinIO  : e275e6a4db0a0adf9263c445ff191bad

[OK] INTEGRITAS DATA TERJAGA 100%! Checksum cocok.

   Record lokal : 10,000
   Record MinIO : 10,000
   Match        : [OK] Ya


---
## Summary: Hasil Data Ingestion — Bronze Layer

In [29]:
print('=' * 65)
print('RINGKASAN DATA INGESTION — BRONZE LAYER')
print('=' * 65)
print(f"""
Medallion Architecture: BRONZE LAYER
   Bronze = Data mentah tanpa transformasi (single source of truth)

Sumber Data:
   Dataset  : Marketing Campaign Performance Dataset
   Sumber   : Kaggle ({KAGGLE_DATASET})
   Format   : CSV
   Record   : {len(df):,} baris
   Kolom    : {len(df.columns)} atribut

Storage (MinIO Data Lake — Bronze Layer):
   Endpoint : {MINIO_ENDPOINT}
   Bucket   : {MINIO_BUCKET}
   Path     : s3a://{MINIO_BUCKET}/{MINIO_BRONZE_CSV}
   
Struktur Medallion di Bucket:
   {MINIO_BUCKET}/
   ├── bronze/     <- CSV mentah        [UPLOADED ✓]
   ├── silver/     <- Cleaned & Features [menunggu Notebook 02]
   ├── gold/       <- Agregasi bisnis    [menunggu Notebook 03-04]
   └── models/     <- Model ML           [menunggu Notebook 03]

[OK] Integritas : Data 100% terjaga (MD5 checksum verified)
[OK] Status     : BRONZE LAYER INGESTION BERHASIL

--> Next Step   : Jalankan Notebook 02 (Bronze → Silver Processing & EDA)
""")
print('=' * 65)

RINGKASAN DATA INGESTION — BRONZE LAYER

Medallion Architecture: BRONZE LAYER
   Bronze = Data mentah tanpa transformasi (single source of truth)

Sumber Data:
   Dataset  : Marketing Campaign Performance Dataset
   Sumber   : Kaggle (mirzayasirabdullah07/marketing-campaign-performance-dataset)
   Format   : CSV
   Record   : 10,000 baris
   Kolom    : 11 atribut

Storage (MinIO Data Lake — Bronze Layer):
   Endpoint : localhost:9000
   Bucket   : marketing-data
   Path     : s3a://marketing-data/bronze/marketing_campaign_performance.csv

Struktur Medallion di Bucket:
   marketing-data/
   ├── bronze/     <- CSV mentah        [UPLOADED ✓]
   ├── silver/     <- Cleaned & Features [menunggu Notebook 02]
   ├── gold/       <- Agregasi bisnis    [menunggu Notebook 03-04]
   └── models/     <- Model ML           [menunggu Notebook 03]

[OK] Integritas : Data 100% terjaga (MD5 checksum verified)
[OK] Status     : BRONZE LAYER INGESTION BERHASIL

--> Next Step   : Jalankan Notebook 02 (Bronze